# Python 函数进阶 + 异常处理 + 文件 IO（练习）

## 1. 默认参数 + 关键字参数


In [ ]:
# 默认参数：调用时不传就用默认值
def greet(name, greeting="你好"):
    return f"{greeting}, {name}"

print(greet("Alice")) 
print(greet("Bob", "Hello"))
print(greet("Charlie", greeting="Hi")) # 关键字传参，顺序无所谓

你好, Alice
Hello, Bob
Hi, Charlie


In [1]:
# 默认参数的经典大坑：可变对象当默认值
def add_item_bad(item, lst=[]): # 坑！默认的[]只创建一次
    lst.append(item)
    return lst

print(add_item_bad("a")) # ['a']
print(add_item_bad("b")) # ['a', 'b'] ← 居然带着上次的！

['a']
['a', 'b']


为什么:默认值 [] 在函数定义时只创建一次,之后所有调用共享同一个 list。这呼应你 Day 1 笔记「list 是引用类型」。

正确写法——默认值用 None,函数内部再创建:

In [3]:
def add_item_good(item, lst=None):
    if lst is None:
        lst = [] # 每次调用都新建
    lst.append(item)
    return lst

print(add_item_good("a"))
print(add_item_good("b")) # ← 正确，互不影响

['a']
['b']


铁律:默认参数永远不要用可变对象([]、{}、set()),要用就写 None 再在函数内新建。高频面试题。

## 2. *args和**kwargs

In [4]:
# *args：把多余的“位置参数”打包成一个tuple
def my_sum(*args):
    print(f"args 是： {args}， 类型：{type(args)}")
    return sum(args)

print(my_sum(1, 2, 3)) # args是（1,2,3）
print(my_sum(1, 2, 3, 4, 5)) # 传几个都行

args 是： (1, 2, 3)， 类型：<class 'tuple'>
6
args 是： (1, 2, 3, 4, 5)， 类型：<class 'tuple'>
15


In [5]:
# **kwargs：把多余的“关键字参数”打包成一个dict
def make_profile(**kwargs):
    print(f"kwargs 是： {kwargs}")
    for k, v in kwargs.items():
        print(f" {k}: {v}")

make_profile(name="Alice", age=30, city="London")

kwargs 是： {'name': 'Alice', 'age': 30, 'city': 'London'}
 name: Alice
 age: 30
 city: London


In [6]:
# 三者混用，顺序固定：普通参数 → *args → **kwargs
def func(a, b, *args, **kwargs):
    print(f"a={a}, b={b}, args={args}, kwargs={kwargs}")

func(1, 2, 3, 4, x=10, y=20)

a=1, b=2, args=(3, 4), kwargs={'x': 10, 'y': 20}


反向用法——拆包:* 和 ** 也能在「调用时」把容器拆开:

In [ ]:
nums = [1, 2, 3]
print(my_sum(*nums)) # 等价于my_sum(1,2,3)

info = {"name": "Bob", "age": 25}
make_profile(**info) # 等价于 make_profile(name="Bob", age=25)

args 是： (1, 2, 3)， 类型：<class 'tuple'>
6
kwargs 是： {'name': 'Bob', 'age': 25}
 name: Bob
 age: 25


*args/**kwargs 你以后会天天见——Pandas、装饰器、各种库的函数签名里全是。现在理解「打包/拆包」这个动作就够了。

## 3. 类型注解

In [10]:
# 给参数和返回值标注类型——不强制，但让代码更清晰、IDE能提示
def calc_total(price:float, quantity:int) -> float:
    return price * quantity

print(calc_total(9.9, 3))

29.700000000000003


In [11]:
# 复杂类型
def process(names: list[str], config: dict[str, int]) -> tuple[int, str]:
    return len(names), names[0]

# 可能是None的：
def find_user(uid: int) -> str | None: # 返回str或None
    return "Alice" if uid == 1 else None

类型注解不影响运行（传错类型Python不报错），它是给人和工具看的文档。数据岗代码里写注解是专业习惯，面试看代码也加分。

## 4. 装饰器基础

装饰器 = 「在不改动原函数的前提下,给它加功能」。先理解「函数可以被当成参数传来传去」:

In [12]:
# 函数本身也是对象，可以赋值、可以当参数
def shout(text):
    return text.upper()

f = shout # 把函数赋给变量
print(f("hello"))

HELLO


In [13]:
# 装饰器：一个“接收函数、返回新函数”的函数
import time

def timer(func):
    def wrapper(*args, **kwargs): # 用*args/**kwargs接住原函数的任意参数
        start = time.time()
        result = func(*args, **kwargs) # 调用原函数
        print(f"{func.__name__} 耗时 {time.time() - start:.4f}s")
        return result
    return wrapper

@timer # @time = slow_func = time(slow_func)
def slow_func(n):
    return sum(range(n))

slow_func(1_000_000) # 自动打印耗时

slow_func 耗时 0.0304s


499999500000

@timer 的含义:slow_func = timer(slow_func)——把 slow_func 喂给 timer,换回一个「带计时功能的新函数」。

装饰器现在理解到「能给函数包一层额外行为」即可,不用写得很溜。它是 Step 2 *args/**kwargs 的最大应用场景——wrapper 要接住原函数的任意参数,就靠它俩。

## 5. 异常处理 + 文件IO

In [14]:
# try/except：把可能出错的代码包起来，出错不让程序崩
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("除数不能为0")
        return None
    
print(safe_divide(10, 2)) 
print(safe_divide(10, 0))

5.0
除数不能为0
None


In [18]:
# 多种异常 + finally
def parse_int(s):
    try:
        return int(s)
    except ValueError:
        print(f"'{s}'不是合法整数")
        return None
    except TypeError:
        print(f"类型不对")
        return None
    finally:
        print("解析尝试结束") # finally 无论成功失败都执行

parse_int("123")

解析尝试结束


123

In [19]:
parse_int("abc")

'abc'不是合法整数
解析尝试结束


文件IO —— 永远用with：

In [20]:
# 写文件
with open("test.txt", "w", encoding="utf-8") as f:
    f.write("第一行\n")
    f.write("第二行\n")
# with块结束，文件自动关闭——不用手动f.close()

# 读文件
with open("test.txt", "r", encoding="utf-8") as f:
    content = f.read()
print(content)

第一行
第二行



In [21]:
# 读写JSON
import json

data = {"name": "Alice", "scores": [90, 85, 95]}
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded)

{'name': 'Alice', 'scores': [90, 85, 95]}


with 的意义:它保证文件一定会被关闭,哪怕中间报错。这其实就是「自动版的 try/finally」。Windows 上 encoding="utf-8" 务必写——不写默认编码会让中文乱码。
